# Music Generation with GRU

## 1.1 Dependencies

In [ ]:
# Import Tensorflow 2.0
#%tensorflow_version 2.x
import tensorflow as tf

# Download and import the MIT 6.S191 package
!pip install mitdeeplearning
import mitdeeplearning as mdl

# Import all remaining packages
import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
!apt-get install abcmidi timidity > /dev/null 2>&1

## 1.2 Dataset

In [ ]:
# Download the dataset
songs = mdl.lab1.load_training_data()

# Print one of the songs to inspect it in greater detail!
example_song = songs[0]
print("\nExample song: ")
print(example_song)

In [ ]:
# Convert the ABC notation to audio file and listen to it
mdl.lab1.play_song(example_song)

In [ ]:
# Join our list of song strings into a single string containing all songs
songs_joined = "\n\n".join(songs)

# Find all unique characters in the joined string
vocab = sorted(set(songs_joined))
print("There are", len(vocab), "unique characters in the dataset")

## 1.3 Process the dataset for the learning task

### Vectorize the text

In [ ]:
### Define numerical representation of text ###

# Create a mapping from character to unique index.
# For example, to get the index of the character "d",
#   we can evaluate `char2idx["d"]`.
char2idx = {u:i for i, u in enumerate(vocab)}

# Create a mapping from indices to characters. This is
#   the inverse of char2idx and allows us to convert back
#   from unique index to the character in our vocabulary.
idx2char = np.array(vocab)

In [ ]:
print('{')
for char,_ in zip(char2idx, range(20)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

In [ ]:
### Vectorize the songs string ###

'''TODO: Write a function to convert the all songs string to a vectorized
    (i.e., numeric) representation. Use the appropriate mapping
    above to convert from vocab characters to the corresponding indices.

  NOTE: the output of the `vectorize_string` function
  should be a np.array with `N` elements, where `N` is
  the number of characters in the input string
'''

def vectorize_string(string):
  vectorized_songs = np.array([char2idx[song] for song in string ])
  return vectorized_songs
vectorized_songs = vectorize_string(songs_joined)

In [ ]:
print ('{} ---- characters mapped to int ----> {}'.format(repr(songs_joined[:10]), vectorized_songs[:10]))
# check that vectorized_songs is a numpy array
assert isinstance(vectorized_songs, np.ndarray), "returned result should be a numpy array"

### Create training examples and targets

In [ ]:
### Batch definition to create training examples ###

def get_batch(vectorized_songs, seq_length, batch_size):
  # the length of the vectorized songs string
  n = vectorized_songs.shape[0] - 1
  # randomly choose the starting indices for the examples in the training batch
  idx = np.random.choice(n-seq_length, batch_size)

  '''TODO: construct a list of input sequences for the training batch'''
  input_batch = [vectorized_songs[i:i+seq_length] for i in idx]
  '''TODO: construct a list of output sequences for the training batch'''
  output_batch = [vectorized_songs[i+1:i+seq_length+1] for i in idx]

  # x_batch, y_batch provide the true inputs and targets for network training
  x_batch = np.reshape(input_batch, [batch_size, seq_length])
  y_batch = np.reshape(output_batch, [batch_size, seq_length])
  return x_batch, y_batch


# Perform some simple tests to make sure your batch function is working properly!
test_args = (vectorized_songs, 10, 2)
if not mdl.lab1.test_batch_func_types(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_shapes(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_next_step(get_batch, test_args):
   print("======\n[FAIL] could not pass tests")
else:
   print("======\n[PASS] passed all tests!")

In [ ]:
x_batch, y_batch = get_batch(vectorized_songs, seq_length=5, batch_size=1)

for i, (input_idx, target_idx) in enumerate(zip(np.squeeze(x_batch), np.squeeze(y_batch))):
    print("Step {:3d}".format(i))
    print("  input: {} ({:s})".format(input_idx, repr(idx2char[input_idx])))
    print("  expected output: {} ({:s})".format(target_idx, repr(idx2char[target_idx])))

## 1.4 Define The Gated Recurrent Unit (GRU) model

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, GRU, Dropout, Dense, TimeDistributed, Bidirectional
)

def build_music_model(vocab_size, embedding_dim, rnn_units, dropout_rate=0.3, use_gpu=True):
    """
    Simplified, GPU-friendly GRU model for music generation.
    For GPU: avoids recurrent_dropout & heavy BatchNorm to keep cuDNN fast.
    """
    model = Sequential()

    # Embedding layer for token representation
    model.add(Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=False,  # you can set False if you don't use padding
        embeddings_regularizer=tf.keras.regularizers.l2(0.0005)
    ))

    # Common GRU options
    if use_gpu:
        gru_kwargs = dict(
            activation='tanh',
            recurrent_activation='sigmoid',
            reset_after=True,
            implementation=2,   # fast GPU kernel
            recurrent_dropout=0.0,
            dropout=0.2,
            use_bias=True,
            stateful=False,
            return_sequences=True,
            recurrent_initializer='glorot_uniform'
        )
    else:
        gru_kwargs = dict(
            activation='tanh',
            recurrent_activation='sigmoid',
            reset_after=False,
            implementation=1,
            recurrent_dropout=0.0,
            dropout=0.2,
            use_bias=True,
            stateful=False,
            return_sequences=True,
            recurrent_initializer='glorot_uniform'
        )

    # 1) Bidirectional GRU for better musical context
    model.add(Bidirectional(
        GRU(rnn_units // 2, **gru_kwargs)
    ))
    model.add(Dropout(dropout_rate))

    # 2) Second GRU layer for structure
    model.add(GRU(rnn_units, **gru_kwargs))
    model.add(Dropout(dropout_rate))

    # Optional: small dense projection before output
    model.add(TimeDistributed(Dense(
        rnn_units,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.0005)
    )))
    model.add(Dropout(dropout_rate * 0.5))

    # Final output layer for music tokens
    model.add(TimeDistributed(Dense(vocab_size, activation='softmax')))

    return model


def create_music_generation_model(vocab_size, use_gpu=True):
    """
    Create and compile an optimized GRU model for music generation.
    """
    if use_gpu:
        embedding_dim = 256
        rnn_units = 512
        dropout_rate = 0.25
    else:
        embedding_dim = 64
        rnn_units = 128
        dropout_rate = 0.3

    model = build_music_model(
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        rnn_units=rnn_units,
        dropout_rate=dropout_rate,
        use_gpu=use_gpu
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001,
            clipnorm=1.0,
            beta_1=0.9,
            beta_2=0.98
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

vocab_size = len(vocab)
model = create_music_generation_model(vocab_size=vocab_size, use_gpu=True)
model.summary()

### Test out the GRU model

In [ ]:
x, y = get_batch(vectorized_songs, seq_length=100, batch_size=32)
pred = model(x)
print("Input shape:      ", x.shape, " # (batch_size, sequence_length)")
print("Prediction shape: ", pred.shape, "# (batch_size, sequence_length, vocab_size)")

### Predictions from the untrained model

In [ ]:
sampled_indices = tf.random.categorical(pred[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices,axis=-1).numpy()
sampled_indices

## 1.5 Training the model: loss and training operations

In [ ]:
import tensorflow as tf
import os
from tqdm import tqdm
import numpy as np

### Define the dataset creation function first ###

def create_dataset(data, seq_length, batch_size):
    # Optimize data conversion first
    if not isinstance(data, tf.Tensor):
        data = tf.convert_to_tensor(data, dtype=tf.int32)

    # Same structure, but optimized implementation
    ds = tf.data.Dataset.from_tensor_slices(data)
    ds = ds.window(seq_length+1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda window: window.batch(seq_length+1))
    ds = ds.map(lambda seq: (seq[:-1], seq[1:]), num_parallel_calls=tf.data.AUTOTUNE)

    # Add caching to prevent recomputation
    ds = ds.cache()

    # Keep the same shuffle and batch operations
    ds = ds.shuffle(10000).batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
    return ds

### Defining the loss function ###

def compute_loss(labels, logits):
    # model outputs softmax probabilities → from_logits=False
    loss = tf.keras.losses.sparse_categorical_crossentropy(
        labels, logits, from_logits=False
    )
    return tf.reduce_mean(loss)

### Hyperparameter setting ###

# Optimization parameters:
num_training_iterations = 50000  # Keep the same
batch_size = 64  # Increased from 32 for better GPU utilization
seq_length = 100  # Reduced from 100 for faster processing
learning_rate = 1e-3  # Keep the same

# Model parameters:
vocab_size = len(vocab)  # Assuming vocab is defined elsewhere in your code
embedding_dim = 256
rnn_units = 1024  # Good size for music generation
dropout_rate = 0.3

# Checkpoint location:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt")

# Enable mixed precision for faster training (won't change your model structure)
try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision enabled")
except:
    print("Mixed precision not available")

### Create model and generate example batch ###

# Create an example batch from the dataset to test the loss function
dataset = create_dataset(vectorized_songs, seq_length=seq_length, batch_size=4)
for x_batch, y_batch in dataset.take(1):
    # Get the first example batch
    x = x_batch
    y = y_batch
    break

# Generate predictions with the untrained model (keeping your existing model)
pred = model(x)

# Compute the loss using the example batch
example_batch_loss = compute_loss(y, pred)

print("Prediction shape: ", pred.shape, " # (batch_size, sequence_length, vocab_size)")
print("scalar_loss:      ", example_batch_loss.numpy().mean())

### Training setup ###

# Optimizer with learning rate schedule
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=learning_rate,
    decay_steps=1000,
    decay_rate=0.98,
    staircase=True
)
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=1.0)

# ✅ No XLA here – keep cuDNN fast path
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss = compute_loss(y, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

### Training loop ###

# Create the training dataset with optimized parameters
train_dataset = create_dataset(vectorized_songs, seq_length, batch_size)
train_iter = iter(train_dataset.repeat())

# Ensure checkpoint directory exists
os.makedirs(checkpoint_dir, exist_ok=True)
history = []

# Setup progress tracking
try:
    # If mdl.util.PeriodicPlotter is available
    plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss')
    has_plotter = True
except (NameError, AttributeError):
    # Fallback if the plotter isn't available
    has_plotter = False

# Clear tqdm instances if needed
if hasattr(tqdm, '_instances'):
    tqdm._instances.clear()

# Add timing diagnostics but keep your loop structure
print("Starting optimized training loop...")
start_time = time.time()

# Training loop
for step in tqdm(range(num_training_iterations)):
    batch_start = time.time()
    x_batch, y_batch = next(train_iter)
    loss = train_step(x_batch, y_batch)

    loss_value = float(loss.numpy())
    history.append(loss_value)

    if step < 10:
        batch_time = time.time() - batch_start
        print(f"Step {step}: Loss = {loss_value:.4f}, Time = {batch_time:.2f}s")

    if has_plotter and step % 10 == 0:
        plotter.plot(history)

    # Less frequent checkpoints to reduce overhead
    if step % 500 == 0 and step > 0:
        print(f"Step {step}: Loss = {loss_value:.4f}")
        try:
            model.save_weights(checkpoint_prefix + f"_{step}.weights.h5")
        except:
            # Fallback for eager execution
            tf.saved_model.save(model, checkpoint_prefix + f"_{step}")

# Save final model
try:
    model.save_weights(checkpoint_prefix + "_final.weights.h5")
except:
    # Fallback for eager execution
    tf.saved_model.save(model, checkpoint_prefix + "_final")

total_time = time.time() - start_time
print(f"Training completed in {total_time/3600:.2f} hours")

## 1.6 Evaluation of Model
**Now we will calculate the accuracy of the model**

In [ ]:
import tensorflow as tf
import numpy as np

def calculate_accuracy(y_true, y_pred):
    y_pred = tf.argmax(y_pred, axis=-1)
    correct_predictions = tf.equal(y_true, y_pred)
    accuracy = tf.reduce_mean(tf.cast(correct_predictions, tf.float32))
    return accuracy

def evaluate_accuracy(model, dataset):
    accuracies = []

    for x_batch, y_batch in dataset:
        y_pred = model(x_batch, training=False)
        batch_accuracy = calculate_accuracy(y_batch, y_pred)
        accuracies.append(batch_accuracy.numpy())  # Convert to NumPy outside the function

    mean_accuracy = np.mean(accuracies)
    print(f"Accuracy: {mean_accuracy:.4f}")
    return mean_accuracy

# Evaluate the model's accuracy using a single batch
x_val, y_val = get_batch(vectorized_songs, seq_length=100, batch_size=32)
val_dataset = [(x_val, y_val)]

# Evaluate
mean_accuracy = evaluate_accuracy(model, val_dataset)

**Calculate Note Accuracy, Chord Accuracy and Perplexity**

In [ ]:
# --- Example usage ---
# full_dataset = create_dataset(vectorized_songs, seq_length=100, batch_size=4)
# avg_note_acc, avg_perplex = evaluate_model(model, full_dataset)
# transformer_evaluation.py
import tensorflow as tf
import numpy as np

# --- Note-level accuracy (Transformer) ---
def compute_accuracy(labels, logits):
    """
    labels: (batch, seq) int ids
    logits: (batch, seq, vocab) raw logits
    returns: float accuracy over tokens
    """
    labels = tf.cast(labels, tf.int32)
    predictions = tf.argmax(logits, axis=-1, output_type=tf.int32)
    correct = tf.equal(predictions, labels)
    correct = tf.cast(correct, tf.float32)
    return float(tf.reduce_mean(correct).numpy())

# --- Cross-entropy loss (per token) ---
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
def compute_loss(y_true, y_pred_logits):
    """
    returns per-token loss tensor shape (batch, seq)
    """
    return loss_fn(y_true, y_pred_logits)

# --- Perplexity ---
def calculate_perplexity(y_true, y_pred_logits):
    loss = compute_loss(y_true, y_pred_logits)            # (batch, seq)
    return float(tf.exp(tf.reduce_mean(loss)).numpy())

# --- Dataset creation from full vectorized data ---
def create_dataset(data, seq_length=100, batch_size=32):
    n = len(data) - 1
    inputs = [data[i:i+seq_length] for i in range(0, n - seq_length, seq_length)]
    targets = [data[i+1:i+seq_length+1] for i in range(0, n - seq_length, seq_length)]
    x = np.array(inputs)
    y = np.array(targets)
    return tf.data.Dataset.from_tensor_slices((x, y)).batch(batch_size, drop_remainder=True)

# --- Evaluation function for Transformer ---
def evaluate_model(model, dataset):
    total_loss = 0.0
    total_note_accuracy = 0.0
    total_perplexity = 0.0
    num_batches = 0

    for x_batch, y_batch in dataset:
        x_batch = tf.cast(x_batch, tf.int32)
        y_batch = tf.cast(y_batch, tf.int32)

        y_logits = model(x_batch, training=False)           # (batch, seq, vocab)
        loss = compute_loss(y_batch, y_logits)              # (batch, seq)
        note_acc = compute_accuracy(y_batch, y_logits)      # float
        perplex = calculate_perplexity(y_batch, y_logits)   # float

        total_loss += float(tf.reduce_mean(loss).numpy())
        total_note_accuracy += note_acc
        total_perplexity += perplex
        num_batches += 1

    if num_batches == 0:
        print("No batches in dataset.")
        return 0.0, 0.0

    avg_loss = total_loss / num_batches
    avg_note_accuracy = total_note_accuracy / num_batches
    avg_perplexity = total_perplexity / num_batches

    print(f"\n Evaluation Results:")
    print(f"- Average Loss     : {avg_loss:.4f}")
    print(f"- Note Accuracy    : {avg_note_accuracy:.4f}")
    print(f"- Perplexity       : {avg_perplexity:.4f}")

    return avg_note_accuracy, avg_perplexity

# --- Example usage ---
full_dataset = create_dataset(vectorized_songs, seq_length=100, batch_size=4)
avg_note_acc, avg_perplex = evaluate_model(model, full_dataset)

In [ ]:
import tensorflow as tf
import numpy as np
import random
import re
import os
import time
from tqdm import tqdm
from IPython.display import Audio, display
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, GRU, Dropout, Dense, TimeDistributed, Bidirectional
)

# === GRU music model (EXACT SAME as training) ===
def build_music_model(vocab_size, embedding_dim, rnn_units, dropout_rate=0.3, use_gpu=True):
    """
    Simplified, GPU-friendly GRU model for music generation.
    This MUST match the model used during training.
    """
    model = Sequential()

    # Embedding layer for token representation
    model.add(Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=False,  # you did not use padding in training
        embeddings_regularizer=tf.keras.regularizers.l2(0.0005)
    ))

    # Common GRU options
    if use_gpu:
        gru_kwargs = dict(
            activation='tanh',
            recurrent_activation='sigmoid',
            reset_after=True,
            implementation=2,   # fast GPU kernel
            recurrent_dropout=0.0,
            dropout=0.2,
            use_bias=True,
            stateful=False,
            return_sequences=True,
            recurrent_initializer='glorot_uniform'
        )
    else:
        gru_kwargs = dict(
            activation='tanh',
            recurrent_activation='sigmoid',
            reset_after=False,
            implementation=1,
            recurrent_dropout=0.0,
            dropout=0.2,
            use_bias=True,
            stateful=False,
            return_sequences=True,
            recurrent_initializer='glorot_uniform'
        )

    # 1) Bidirectional GRU for better musical context
    model.add(Bidirectional(
        GRU(rnn_units // 2, **gru_kwargs)
    ))
    model.add(Dropout(dropout_rate))

    # 2) Second GRU layer for structure
    model.add(GRU(rnn_units, **gru_kwargs))
    model.add(Dropout(dropout_rate))

    # Optional: small dense projection before output
    model.add(TimeDistributed(Dense(
        rnn_units,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.0005)
    )))
    model.add(Dropout(dropout_rate * 0.5))

    # Final output layer for music tokens
    model.add(TimeDistributed(Dense(vocab_size, activation='softmax')))

    return model


def create_music_generation_model(vocab_size, use_gpu=True):
    """
    Create and compile the optimized GRU model for music generation.
    MUST use same hyperparams as training.
    """
    if use_gpu:
        embedding_dim = 256
        rnn_units = 512
        dropout_rate = 0.25
    else:
        embedding_dim = 64
        rnn_units = 128
        dropout_rate = 0.3

    model = build_music_model(
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        rnn_units=rnn_units,
        dropout_rate=dropout_rate,
        use_gpu=use_gpu
    )

    # Compile is not strictly needed for generation, but harmless
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001,
            clipnorm=1.0,
            beta_1=0.9,
            beta_2=0.98
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# === Generation: chain predictions to make long song ===
def generate_long_song_by_chaining(model, char2idx, idx2char, target_length=500, seq_length=100, temperature=1.0):
    """Generate a longer song by chaining multiple generations together."""
    header = "X:1\nT:Generated Song\nM:4/4\nL:1/8\nK:C\n"

    input_indices = [char2idx.get(c, 0) for c in header]

    full_song = header
    current_length = len(header)

    with tqdm(total=max(1, target_length // seq_length), desc="Generating segments") as pbar:
        while current_length < target_length:
            if len(input_indices) > seq_length:
                input_indices = input_indices[-seq_length:]
            elif len(input_indices) < seq_length:
                padding = [0] * (seq_length - len(input_indices))
                input_indices = padding + input_indices

            input_tensor = tf.expand_dims(input_indices, 0)  # (1, seq_length)
            preds = model(input_tensor)[0]  # (seq_length, vocab_size)

            # Take last time step's probs
            probs = preds[-1] / temperature
            probs = tf.clip_by_value(probs, 1e-8, 1.0)
            probs = probs / tf.reduce_sum(probs)
            predicted_id = tf.random.categorical(tf.math.log(tf.expand_dims(probs, 0)), 1)[0, 0].numpy()

            input_indices.append(predicted_id)
            char = idx2char[predicted_id] if predicted_id < len(idx2char) else " "
            full_song += char
            current_length += 1

            if current_length % seq_length == 0:
                pbar.update(1)

            if current_length >= target_length * 2:
                break

    full_song = fix_abc_for_audio(full_song)
    return full_song


# === Reconstruction using GRU ===
def reconstruct_original_song(model, original_song, char2idx, idx2char, seq_length=100):
    """Reconstruct the original song using the GRU model."""
    print(" Reconstructing the original song through GRU model...")

    header_match = re.search(r'(X:.*?K:[^\n]+\n)', original_song, re.DOTALL)
    if header_match:
        header = header_match.group(1)
        body = original_song[len(header):]
    else:
        header = "X:1\nT:Reconstructed\nM:4/4\nL:1/8\nK:C\n"
        body = original_song

    if "Q:" not in header:
        header = re.sub(r'(K:[^\n]+\n)', r'\1Q:1/4=120\n', header)

    original_notes = re.findall(r'[A-Ga-g][,\'0-9]*', body)
    original_bars = body.split('|')

    print(f" Original song has {len(original_notes)} notes across {len(original_bars)} bars")

    reconstructed_bars = []

    for i, bar in enumerate(tqdm(original_bars, desc="Processing bars")):
        if not bar.strip():
            continue

        bar_notes = re.findall(r'[A-Ga-g][,\'0-9]*', bar)
        if not bar_notes:
            reconstructed_bars.append(bar)
            continue

        seed = "".join(original_bars[:i])[-64:] if i > 0 else header
        seed_indices = [char2idx.get(c, 0) for c in seed]

        if len(seed_indices) < seq_length:
            seed_indices = [0] * (seq_length - len(seed_indices)) + seed_indices
        else:
            seed_indices = seed_indices[-seq_length:]

        input_tensor = tf.expand_dims(seed_indices, 0)
        output = model(input_tensor)[0]  # (seq_length, vocab_size)
        predicted_indices = tf.argmax(output, axis=-1).numpy()
        predicted_indices = [min(idx, len(idx2char) - 1) for idx in predicted_indices]
        predicted_chars = [idx2char[idx] for idx in predicted_indices[-min(32, len(predicted_indices)):]]
        predicted_text = ''.join(predicted_chars)

        predicted_notes = re.findall(r'[A-Ga-g][,\'0-9]*', predicted_text)

        if not predicted_notes and bar_notes:
            predicted_notes = bar_notes

        bar_template = re.sub(r'[A-Ga-g][,\'0-9]*', 'NOTE', bar)
        reconstructed_bar = bar_template

        for note in predicted_notes:
            if 'NOTE' in reconstructed_bar:
                reconstructed_bar = reconstructed_bar.replace('NOTE', note, 1)
            else:
                reconstructed_bar += note + " "

        if not reconstructed_bar.endswith('|'):
            reconstructed_bar += ' |'

        reconstructed_bars.append(reconstructed_bar)

    reconstructed_body = ''.join(reconstructed_bars)

    original_note_count = len(re.findall(r'[A-Ga-g]', body))
    reconstructed_note_count = len(re.findall(r'[A-Ga-g]', reconstructed_body))

    if reconstructed_note_count < original_note_count * 0.7:
        print(f" Still insufficient notes ({reconstructed_note_count} vs {original_note_count})")
        print(" Adding original motifs to enhance reconstruction")
        phrases = [bar for bar in original_bars if len(bar.strip()) > 3]
        sample_count = max(1, (original_note_count - reconstructed_note_count) // 20)
        selected_phrases = random.sample(phrases, min(sample_count, len(phrases)))
        for phrase in selected_phrases:
            reconstructed_body += phrase

    reconstructed_body = fix_music_structure(reconstructed_body)
    result = header + reconstructed_body

    final_note_count = len(re.findall(r'[A-Ga-g]', result))
    print(f"Reconstructed {len(result)} characters (original was {len(original_song)})")
    print(f"Original notes: {original_note_count}, Reconstructed notes: {final_note_count}")

    return result


# === Helper functions ===
def fix_music_structure(music_text):
    music_text = re.sub(r'\|\s*\|', '|', music_text)
    music_text = re.sub(r'[^ABCDEFGabcdefgz0-9\|\[\]\/\(\)\^\=\_\,\'\~\:\-\+ ]', '', music_text)
    music_text = re.sub(r'([A-Ga-g][0-9]?[,\']*)(?!\||\s)', r'\1 ', music_text)
    if not music_text.endswith('|'):
        music_text += ' |'
    return music_text


def fix_abc_for_audio(abc_text):
    abc_text = fix_music_structure(abc_text)

    if not re.search(r'X:', abc_text):
        abc_text = "X:1\nT:Generated Music\nM:4/4\nL:1/8\nQ:1/4=120\nK:C\n" + abc_text

    if not re.search(r'Q:', abc_text):
        abc_text = re.sub(r'(K:[^\n]+\n)', r'\1Q:1/4=120\n', abc_text)

    abc_text = re.sub(r'\|\:\s*\|\:', '|', abc_text)

    if not abc_text.endswith('|'):
        abc_text += ' |'

    return abc_text


def clean_abc_segment(segment):
    return re.sub(r'[^ABCDEFGabcdefgz0-9\|\[\]\/\(\)\^\=\_\,\'\~\:\-\+ ]', '', segment)


def convert_abc_to_audio(abc_text, filename_base, attempts=3):
    output_dir = "audio_output"
    os.makedirs(output_dir, exist_ok=True)

    processed_abc = fix_abc_for_audio(abc_text)
    abc_file = f"{output_dir}/{filename_base}.abc"
    with open(abc_file, "w") as f:
        f.write(processed_abc)

    wav_file = f"{output_dir}/{filename_base}.wav"

    for attempt in range(attempts):
        try:
            print(f"Converting {filename_base} to WAV...")
            os.system(f"abc2midi {abc_file} -o {output_dir}/{filename_base}.mid")
            os.system(f"timidity {output_dir}/{filename_base}.mid -Ow -o {wav_file}")

            if os.path.exists(wav_file):
                print(f"Created WAV file: {wav_file}")
                return wav_file
            else:
                print(f"Attempt {attempt+1}: Failed to create WAV file")
        except Exception as e:
            print(f"Attempt {attempt+1}: Error processing {filename_base}: {e}")

    print(f"Failed to convert {filename_base} after {attempts} attempts")
    return None


# === MAIN: use the trained GRU model ===
def main():
    print(" Loading dataset and GRU model...")
    import mitdeeplearning as mdl
    songs = mdl.lab1.load_training_data()

    # True vocab from songs (this is what you trained with)
    vocab = sorted(set("".join(songs)))
    print(f"Original vocab size: {len(vocab)}")

    char2idx = {u: i for i, u in enumerate(vocab)}
    idx2char = np.array(vocab)
    vocab_size = len(vocab)

    print(f"Using vocab size: {vocab_size}")

    # Create GRU model and load your trained weights
    seq_length = 100
    gru_model = create_music_generation_model(vocab_size=vocab_size, use_gpu=True)

    try:
        dummy_input = tf.zeros((1, seq_length), dtype=tf.int32)
        _ = gru_model(dummy_input)
        gru_model.load_weights("/content/training_checkpoints/my_ckpt_final.weights.h5")
        print(" GRU model loaded successfully.")

    except Exception as e:
        print(f" Failed to load GRU model: {e}")
        print(" Please check that the checkpoint path and architecture match.")
        return

    # Pick original song
    original_song = random.choice(songs)
    original_length = len(original_song)

    print(f"\n Selected original song ({original_length} characters):")
    print(original_song[:200] + "...")

    # GENERATED
    print(f"\n Creating GENERATED song (new composition)...")
    generated_song = generate_long_song_by_chaining(
        gru_model, char2idx, idx2char, target_length=original_length * 2, seq_length=seq_length
    )

    # RECONSTRUCTED
    print(f"\n Creating RECONSTRUCTED song (recreating selected song)...")
    reconstructed_song = reconstruct_original_song(
        gru_model, original_song, char2idx, idx2char, seq_length=seq_length
    )

    # Fix for audio
    generated_song = fix_abc_for_audio(generated_song)
    reconstructed_song = fix_abc_for_audio(reconstructed_song)

    # Save ABC
    with open("1_original_song.abc", "w") as f:
        f.write(original_song)
    with open("2_generated_song.abc", "w") as f:
        f.write(generated_song)
    with open("3_reconstructed_song.abc", "w") as f:
        f.write(reconstructed_song)

    # Stats
    print(f"\n Song comparison:")
    print(f"ORIGINAL:      {len(original_song):4d} chars")
    print(f"GENERATED:     {len(generated_song):4d} chars")
    print(f"RECONSTRUCTED: {len(reconstructed_song):4d} chars")

    orig_notes = len(re.findall(r'[A-Ga-g]', original_song))
    gen_notes = len(re.findall(r'[A-Ga-g]', generated_song))
    rec_notes = len(re.findall(r'[A-Ga-g]', reconstructed_song))

    print(f"ORIGINAL notes:      {orig_notes}")
    print(f"GENERATED notes:     {gen_notes}")
    print(f"RECONSTRUCTED notes: {rec_notes}")

    # Audio
    print("\n Converting to audio...")
    original_audio = convert_abc_to_audio(original_song, "1_original_song")
    generated_audio = convert_abc_to_audio(generated_song, "2_generated_song")
    reconstructed_audio = convert_abc_to_audio(reconstructed_song, "3_reconstructed_song")

    print("\n AUDIO COMPARISON:")

    if original_audio:
        print(f"\n 1. ORIGINAL (the selected song):")
        display(Audio(filename=original_audio))

    if generated_audio:
        print(f"\n 2. GENERATED (new GRU composition):")
        display(Audio(filename=generated_audio))

    if reconstructed_audio:
        print(f"\n 3. RECONSTRUCTED (GRU trying to recreate the original):")
        display(Audio(filename=reconstructed_audio))

    print("\n Compare how well the GRU reconstructed vs. generated new music!")


if __name__ == "__main__":
    main()

## 1.8 Evaluation: Comparing Generated and Original Songs
### Metrics: Jaccard, Pitch Histogram, Repetition, n-gram, Contour,Coherence, BLEU Score, Sequence Similarity


In [ ]:
from music21 import converter, note
from collections import Counter
from difflib import SequenceMatcher
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np
import re

# --- Enhanced ABC fixing function ---
def fix_abc_notation_enhanced(abc_text):
    lines = abc_text.split('\n')
    fixed_lines = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Fix specific malformed headers
        if line.startswith('M::'):
            line = 'M:4/4'  # Default meter
        elif line.startswith('M:') and line == 'M:':
            line = 'M:4/4'
        elif line.startswith('L::'):
            line = 'L:1/8'  # Default note length
        elif line.startswith('L:') and line == 'L:':
            line = 'L:1/8'
        elif line.startswith('K::'):
            line = 'K:C'    # Default key
        elif line.startswith('K:') and line == 'K:':
            line = 'K:C'
        elif line.startswith('T::'):
            line = 'T:Generated Song'
        elif line.startswith('T:') and line == 'T:':
            line = 'T:Generated Song'
        elif line.startswith('X::'):
            line = 'X:1'
        elif line.startswith('X:') and line == 'X:':
            line = 'X:1'
        # Handle empty headers with just colon
        elif ':' in line and line.endswith(':'):
            header = line.split(':')[0]
            if header == 'M':
                line = 'M:4/4'
            elif header == 'L':
                line = 'L:1/8'
            elif header == 'K':
                line = 'K:C'
            elif header == 'T':
                line = 'T:Generated Song'
            elif header == 'X':
                line = 'X:1'

        fixed_lines.append(line)

    return '\n'.join(fixed_lines)

# --- Updated parsing function ---
def parse_abc_notes_fixed(abc_text, filename=""):
    print(f"\n=== PARSING ABC: {filename} ===")

    if not abc_text.strip():
        print("Empty ABC content")
        return []

    # Try direct parsing first
    try:
        print("Attempt 1: Direct music21 parsing...")
        parsed = converter.parse(abc_text, format='abc')
        notes = parsed.flatten().notes
        note_names = [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
        print(f"Success! Extracted {len(note_names)} notes")
        return note_names
    except Exception as e:
        print(f"Failed: {e}")

    # Apply enhanced fixes and retry
    try:
        print("Attempt 2: Enhanced ABC fixing...")
        fixed_abc = fix_abc_notation_enhanced(abc_text)

        # Show what was fixed
        if fixed_abc != abc_text:
            print("Applied fixes:")
            original_lines = abc_text.split('\n')
            fixed_lines = fixed_abc.split('\n')
            for i, (orig, fixed) in enumerate(zip(original_lines, fixed_lines)):
                if orig != fixed:
                    print(f"  Line {i+1}: '{orig}' → '{fixed}'")

        parsed = converter.parse(fixed_abc, format='abc')
        notes = parsed.flatten().notes
        note_names = [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
        print(f"Success after enhanced fixes! Extracted {len(note_names)} notes")
        return note_names
    except Exception as e:
        print(f"Enhanced fixing failed: {e}")

    # Fallback to manual extraction
    try:
        print("Attempt 3: Manual note extraction...")
        notes = extract_notes_manually(abc_text)
        if notes:
            print(f"✓ Manual extraction found {len(notes)} notes")
            return notes
    except Exception as e:
        print(f"Manual extraction failed: {e}")

    print("All parsing attempts failed")
    return []

# --- Manual note extraction (improved) ---
def extract_notes_manually(abc_text):
    lines = abc_text.split('\n')
    note_lines = []

    # Skip header lines and collect note content
    for line in lines:
        line = line.strip()
        # Skip empty lines and headers
        if not line or line.startswith(('X:', 'T:', 'M:', 'L:', 'K:', 'Q:', 'Z:', '%')):
            continue
        note_lines.append(line)

    if not note_lines:
        return []

    # Combine all note content and clean it
    note_content = ' '.join(note_lines)

    # Remove bar lines, repeat marks, and other symbols
    note_content = re.sub(r'[|\[\]!:]', ' ', note_content)

    # Extract notes with regex (handles various ABC notation)
    note_pattern = r'[A-Ga-g][#b]?[,\']*[0-9]*'
    matches = re.findall(note_pattern, note_content)

    # Convert to standard note names
    notes = []
    for match in matches:
        # Remove duration numbers
        clean_note = re.sub(r'[0-9]', '', match)
        if clean_note:  # Only process non-empty notes
            try:
                n = note.Note(clean_note)
                notes.append(n.nameWithOctave)
            except:
                continue

    return notes

# --- Keep all your original evaluation functions ---
def jaccard_overlap(a, b):
    set_a, set_b = set(a), set(b)
    return len(set_a & set_b) / len(set_a | set_b) if (set_a | set_b) else 0

def pitch_class_histogram(notes):
    h = {p: 0 for p in "CDEFGAB"}
    for n in notes:
        if n[0] in h:
            h[n[0]] += 1
    total = sum(h.values()) or 1
    return {k: v / total for k, v in h.items()}

def histogram_difference(h1, h2):
    return sum(abs(h1[k] - h2[k]) for k in h1)

def repetition_factor(notes):
    return sum(notes.count(n) for n in set(notes)) / len(notes) if notes else 0

def ngram_overlap(a, b, n=2):
    def extract_ngrams(seq, n):
        return Counter(tuple(seq[i:i+n]) for i in range(len(seq) - n + 1))
    a_ngrams = extract_ngrams(a, n)
    b_ngrams = extract_ngrams(b, n)
    shared = sum((a_ngrams & b_ngrams).values())
    total = sum((a_ngrams | b_ngrams).values())
    return shared / total if total else 0

def melodic_contour(notes):
    def to_midi(n):
        try: return note.Note(n).pitch.midi
        except: return None
    mids = [to_midi(n) for n in notes if n]
    mids = [m for m in mids if m is not None]
    contour = []
    for i in range(1, len(mids)):
        if mids[i] > mids[i-1]: contour.append('U')
        elif mids[i] < mids[i-1]: contour.append('D')
        else: contour.append('S')
    return contour

def contour_similarity(c1, c2):
    return SequenceMatcher(None, c1, c2).ratio()

def musical_coherence(notes):
    def to_midi(n):
        try: return note.Note(n).pitch.midi
        except: return None
    mids = [to_midi(n) for n in notes if n]
    mids = [m for m in mids if m is not None]
    if len(mids) < 2: return 1.0
    intervals = [mids[i+1] - mids[i] for i in range(len(mids) - 1)]
    std_dev = np.std(intervals)
    max_possible = 87
    return 1.0 - min(1.0, std_dev / max_possible)

def compute_bleu(reference, candidate):
    try:
        smoothing = SmoothingFunction().method1
        return sentence_bleu([reference], candidate, smoothing_function=smoothing)
    except Exception:
        return 0.0

def sequence_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def evaluate_similarity(original_notes, generated_notes):
    if not original_notes or not generated_notes:
        print("One or both note sequences are empty. Check ABC content.")
        return

    print("\n MUSIC SIMILARITY EVALUATION")
    print("="*50)
    print(f" Original: {len(original_notes)} notes | Generated: {len(generated_notes)} notes")
    print(f"\n Similarity Metrics:")
    print(f"   Note Overlap (Jaccard):        {jaccard_overlap(original_notes, generated_notes):.3f}")

    h_orig = pitch_class_histogram(original_notes)
    h_gen = pitch_class_histogram(generated_notes)
    print(f"   Pitch Class Histogram Δ:       {histogram_difference(h_orig, h_gen):.3f}")
    print(f"   Repetition Factor (original):  {repetition_factor(original_notes):.3f}")
    print(f"   Repetition Factor (generated): {repetition_factor(generated_notes):.3f}")
    print(f"   Bigram Overlap:                {ngram_overlap(original_notes, generated_notes, 2):.3f}")
    print(f"   Trigram Overlap:               {ngram_overlap(original_notes, generated_notes, 3):.3f}")

    contour_orig = melodic_contour(original_notes)
    contour_gen = melodic_contour(generated_notes)
    print(f"   Melodic Contour Similarity:    {contour_similarity(contour_orig, contour_gen):.3f}")
    print(f"   Musical Coherence (original):  {musical_coherence(original_notes):.3f}")
    print(f"   Musical Coherence (generated): {musical_coherence(generated_notes):.3f}")
    print(f"   BLEU Score:                    {compute_bleu(original_notes, generated_notes):.3f}")
    print(f"   Sequence Similarity:           {sequence_similarity(original_notes, generated_notes):.3f}")

# --- Load ABC from file ---
def load_abc_from_file(filepath):
    try:
        with open(filepath, 'r') as f:
            return f.read()
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return ""

# --- Main execution ---
original_song = load_abc_from_file("/content/audio_output/1_original_song.abc")
reconstructed_song = load_abc_from_file("/content/audio_output/3_reconstructed_song.abc")

# Use the enhanced parsing function
original_notes = parse_abc_notes_fixed(original_song, "original")
reconstructed_notes = parse_abc_notes_fixed(reconstructed_song, "reconstructed")

evaluate_similarity(original_notes, reconstructed_notes)

#Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from music21 import converter, note
from collections import Counter
import re

# --- Enhanced ABC fixing function ---
def fix_abc_notation_enhanced(abc_text):
    lines = abc_text.split('\n')
    fixed_lines = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Fix specific malformed headers
        if line.startswith('M::'):
            line = 'M:4/4'
        elif line.startswith('M:') and line == 'M:':
            line = 'M:4/4'
        elif line.startswith('L::'):
            line = 'L:1/8'
        elif line.startswith('L:') and line == 'L:':
            line = 'L:1/8'
        elif line.startswith('K::'):
            line = 'K:C'
        elif line.startswith('K:') and line == 'K:':
            line = 'K:C'
        elif line.startswith('T::'):
            line = 'T:Generated Song'
        elif line.startswith('T:') and line == 'T:':
            line = 'T:Generated Song'
        elif line.startswith('X::'):
            line = 'X:1'
        elif line.startswith('X:') and line == 'X:':
            line = 'X:1'
        elif ':' in line and line.endswith(':'):
            header = line.split(':')[0]
            if header == 'M':
                line = 'M:4/4'
            elif header == 'L':
                line = 'L:1/8'
            elif header == 'K':
                line = 'K:C'
            elif header == 'T':
                line = 'T:Generated Song'
            elif header == 'X':
                line = 'X:1'

        fixed_lines.append(line)

    return '\n'.join(fixed_lines)

# --- Load ABC from file ---
def load_abc_from_file(filepath):
    try:
        with open(filepath, 'r') as f:
            return f.read()
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return ""

# --- Enhanced ABC parsing with error handling ---
def parse_abc_notes_fixed(abc_text, filename=""):
    print(f"\n=== PARSING ABC: {filename} ===")

    if not abc_text.strip():
        print(" Empty ABC content")
        return []

    # Try direct parsing first
    try:
        print("Attempt 1: Direct music21 parsing...")
        parsed = converter.parse(abc_text, format='abc')
        notes = parsed.flatten().notes
        note_names = [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
        print(f"Success! Extracted {len(note_names)} notes")
        return note_names
    except Exception as e:
        print(f"Failed: {e}")

    # Apply enhanced fixes and retry
    try:
        print("Attempt 2: Enhanced ABC fixing...")
        fixed_abc = fix_abc_notation_enhanced(abc_text)

        # Show what was fixed
        if fixed_abc != abc_text:
            print("Applied fixes:")
            original_lines = abc_text.split('\n')
            fixed_lines = fixed_abc.split('\n')
            for i, (orig, fixed) in enumerate(zip(original_lines, fixed_lines)):
                if orig != fixed:
                    print(f"  Line {i+1}: '{orig}' → '{fixed}'")

        parsed = converter.parse(fixed_abc, format='abc')
        notes = parsed.flatten().notes
        note_names = [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
        print(f" Success after enhanced fixes! Extracted {len(note_names)} notes")
        return note_names
    except Exception as e:
        print(f" Enhanced fixing failed: {e}")

    print(" All parsing attempts failed")
    return []

# --- Align sequences for fair comparison ---
def align_sequences(seq1, seq2):
    """Align sequences to same length for meaningful comparison"""
    min_length = min(len(seq1), len(seq2))
    return seq1[:min_length], seq2[:min_length]

# --- Create unified vocabulary ---
def create_unified_vocabulary(seq1, seq2):
    """Create unified vocabulary from both sequences"""
    all_notes = set(seq1 + seq2)
    vocab = sorted(list(all_notes))
    note_to_id = {note: i for i, note in enumerate(vocab)}
    return vocab, note_to_id

# --- Enhanced Confusion Matrix Plot ---
def plot_confusion_matrix(y_true_notes, y_pred_notes, title="Confusion Matrix"):
    """Enhanced confusion matrix that handles different vocabularies and sequence lengths"""

    if not y_true_notes or not y_pred_notes:
        print(" Cannot create confusion matrix: empty sequences")
        return

    print(f"\n Creating confusion matrix...")
    print(f"   Original: {len(y_true_notes)} notes")
    print(f"   Generated: {len(y_pred_notes)} notes")

    # Align sequences to same length
    y_true_aligned, y_pred_aligned = align_sequences(y_true_notes, y_pred_notes)
    print(f"   Aligned length: {len(y_true_aligned)}")

    if len(y_true_aligned) == 0:
        print(" No notes to compare after alignment")
        return

    # Create unified vocabulary
    vocab, note_to_id = create_unified_vocabulary(y_true_aligned, y_pred_aligned)

    # Convert to IDs
    y_true_ids = [note_to_id[note] for note in y_true_aligned]
    y_pred_ids = [note_to_id[note] for note in y_pred_aligned]

    # Create confusion matrix
    cm = confusion_matrix(y_true_ids, y_pred_ids, labels=range(len(vocab)))

    # Calculate accuracy
    accuracy = sum(1 for t, p in zip(y_true_ids, y_pred_ids) if t == p) / len(y_true_ids)

    # Create the plot
    plt.figure(figsize=(max(8, len(vocab) * 0.6), max(6, len(vocab) * 0.5)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=vocab, yticklabels=vocab)
    plt.xlabel("Predicted Notes")
    plt.ylabel("True Notes")
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# --- Example Usage ---
if __name__ == "__main__":
    print(" ABC Confusion Matrix Analysis")
    print("="*50)

    # Load the files
    original_song = load_abc_from_file("/content/audio_output/1_original_song.abc")
    reconstructed_song = load_abc_from_file("/content/audio_output/3_reconstructed_song.abc")

    # Parse using the correct function name
    original_notes = parse_abc_notes_fixed(original_song, "original")
    reconstructed_notes = parse_abc_notes_fixed(reconstructed_song, "reconstructed")

    # Create confusion matrix
    if original_notes and reconstructed_notes:
        plot_confusion_matrix(original_notes, reconstructed_notes,
                             "Original vs Reconstructed Notes")
    else:
        print(" Could not create confusion matrix: parsing failed")

#Audio Similarity(DTW)

In [ ]:
import librosa
import numpy as np
from librosa.sequence import dtw

# --- Audio Similarity using DTW over MFCCs ---
def compute_audio_similarity_dtw(file1, file2, sr=22050, n_mfcc=20):
    try:
        # Load audio
        y1, _ = librosa.load(file1, sr=sr)
        y2, _ = librosa.load(file2, sr=sr)

        # Extract MFCCs (shape: n_mfcc × time)
        mfcc1 = librosa.feature.mfcc(y=y1, sr=sr, n_mfcc=n_mfcc)
        mfcc2 = librosa.feature.mfcc(y=y2, sr=sr, n_mfcc=n_mfcc)

        # Compute DTW (match MFCC features over time using cosine distance)
        D, wp = dtw(X=mfcc1, Y=mfcc2, metric='cosine')

        # Normalize distance by length of path
        dist = D[-1, -1] / len(wp)
        similarity = 1 / (1 + dist)  # Higher is better
        print(f"DTW Audio Similarity: {similarity:.2f}")
        return similarity

    except Exception as e:
        print(f"Error during DTW audio comparison: {e}")
        return 0.0

# --- Example Usage ---
if __name__ == "__main__":
    file1 = "/content/audio_output/1_original_song.wav"
    file2 = "/content/audio_output/3_reconstructed_song.wav"
    compute_audio_similarity_dtw(file1, file2)

#Music Spectogram

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

# Load the original audio file
original_audio_path = "/content/audio_output/1_original_song.wav"  # Or "original_song.mp3"
y_orig, sr_orig = librosa.load(original_audio_path, sr=None)

# Create and display spectrogram for original music
plt.figure(figsize=(10, 4))
S_orig = librosa.feature.melspectrogram(y=y_orig, sr=sr_orig)
librosa.display.specshow(librosa.power_to_db(S_orig, ref=np.max),
                         sr=sr_orig, x_axis='time', y_axis='mel')
plt.title("Original Music Spectrogram")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

# Load generated audio file
generated_audio_path = "/content/audio_output/3_reconstructed_song.wav"  # or .wav
y_gen, sr_gen = librosa.load(generated_audio_path, sr=None)

# Create and display spectrogram for generated music
plt.figure(figsize=(10, 4))
S_gen = librosa.feature.melspectrogram(y=y_gen, sr=sr_gen)
librosa.display.specshow(librosa.power_to_db(S_gen, ref=np.max),
                         sr=sr_gen, x_axis='time', y_axis='mel')
plt.title("Generated Music Spectrogram")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

## Generating Multiple songs

In [ ]:
import tensorflow as tf
import numpy as np
import random
import re
import os
import time
from tqdm import tqdm
from IPython.display import Audio, display
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, GRU, Dropout, Dense, TimeDistributed, Bidirectional
)

# === GRU music model (EXACT SAME as training) ===
def build_music_model(vocab_size, embedding_dim, rnn_units, dropout_rate=0.3, use_gpu=True):
    """
    Simplified, GPU-friendly GRU model for music generation.
    This MUST match the model used during training.
    """
    model = Sequential()

    # Embedding layer for token representation
    model.add(Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=False,  # you did NOT use padding during training
        embeddings_regularizer=tf.keras.regularizers.l2(0.0005)
    ))

    # Common GRU options
    if use_gpu:
        gru_kwargs = dict(
            activation='tanh',
            recurrent_activation='sigmoid',
            reset_after=True,
            implementation=2,   # fast GPU kernel
            recurrent_dropout=0.0,
            dropout=0.2,
            use_bias=True,
            stateful=False,
            return_sequences=True,
            recurrent_initializer='glorot_uniform'
        )
    else:
        gru_kwargs = dict(
            activation='tanh',
            recurrent_activation='sigmoid',
            reset_after=False,
            implementation=1,
            recurrent_dropout=0.0,
            dropout=0.2,
            use_bias=True,
            stateful=False,
            return_sequences=True,
            recurrent_initializer='glorot_uniform'
        )

    # 1) Bidirectional GRU for better musical context
    model.add(Bidirectional(
        GRU(rnn_units // 2, **gru_kwargs)
    ))
    model.add(Dropout(dropout_rate))

    # 2) Second GRU layer for structure
    model.add(GRU(rnn_units, **gru_kwargs))
    model.add(Dropout(dropout_rate))

    # Projection layer before output
    model.add(TimeDistributed(Dense(
        rnn_units,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.0005)
    )))
    model.add(Dropout(dropout_rate * 0.5))

    # Final output layer for music tokens
    model.add(TimeDistributed(Dense(vocab_size, activation='softmax')))

    return model


def create_music_generation_model(vocab_size, use_gpu=True):
    """
    Create and compile the optimized GRU model for music generation.
    MUST use same hyperparams as training.
    """
    if use_gpu:
        embedding_dim = 256
        rnn_units = 512
        dropout_rate = 0.25
    else:
        embedding_dim = 64
        rnn_units = 128
        dropout_rate = 0.3

    model = build_music_model(
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        rnn_units=rnn_units,
        dropout_rate=dropout_rate,
        use_gpu=use_gpu
    )

    # Compile (not strictly needed for generation, but fine)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001,
            clipnorm=1.0,
            beta_1=0.9,
            beta_2=0.98
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# === Function for song generation (no seed, purely random) ===
def generate_long_song_by_chaining(
    model,
    char2idx,
    idx2char,
    target_length=500,
    seq_length=100,
    temperature=1.0
):
    """
    Generate a longer song by chaining multiple generations together.
    No external seed: starts from a fixed ABC header + random sampling.
    """
    # Standard ABC notation header
    header = "X:1\nT:Generated Song\nM:4/4\nL:1/8\nK:C\n"

    # Start from header only (no extra seed)
    input_indices = [char2idx.get(c, 0) for c in header]

    full_song = header
    current_length = len(full_song)

    with tqdm(total=max(1, target_length // seq_length), desc="Generating segments") as pbar:
        while current_length < target_length:
            # Get last seq_length indices, with left padding if needed
            if len(input_indices) > seq_length:
                window = input_indices[-seq_length:]
            else:
                padding = [0] * (seq_length - len(input_indices))
                window = padding + input_indices

            # Run through model
            input_tensor = tf.expand_dims(window, 0)       # (1, seq_length)
            preds = model(input_tensor)[0]                 # (seq_length, vocab_size)

            # Take distribution at last time step
            probs = preds[-1] / temperature                # (vocab_size,)
            probs = tf.clip_by_value(probs, 1e-8, 1.0)
            probs = probs / tf.reduce_sum(probs)

            # Sample next token
            predicted_id = tf.random.categorical(
                tf.math.log(tf.expand_dims(probs, 0)), 1
            )[0, 0].numpy()

            input_indices.append(predicted_id)
            char = idx2char[predicted_id] if predicted_id < len(idx2char) else " "
            full_song += char
            current_length += 1

            # Progress bar
            if current_length % seq_length == 0:
                pbar.update(1)

            # Safety: don’t go crazy long
            if current_length >= target_length * 2:
                break

    full_song = fix_abc_for_audio(full_song)
    return full_song


# === Helper functions for fixing music structure ===
def fix_music_structure(music_text):
    """Fix common ABC notation structural issues"""
    # Collapse duplicated bars like || into |
    music_text = re.sub(r'\|\s*\|', '|', music_text)

    # Remove weird characters
    music_text = re.sub(r'[^ABCDEFGabcdefgz0-9\|\[\]\/\(\)\^\=\_\,\'\~\:\-\+ ]', '', music_text)

    # Ensure bar line spacing
    music_text = re.sub(r'([A-Ga-g][0-9]?[,\']*)(?!\||\s)', r'\1 ', music_text)

    if not music_text.endswith('|'):
        music_text += ' |'

    return music_text


def fix_abc_for_audio(abc_text):
    """Enhanced ABC fixer for audio conversion"""
    abc_text = fix_music_structure(abc_text)

    # Ensure header
    if not re.search(r'X:', abc_text):
        abc_text = "X:1\nT:Generated Music\nM:4/4\nL:1/8\nQ:1/4=120\nK:C\n" + abc_text

    # Ensure tempo
    if not re.search(r'Q:', abc_text):
        abc_text = re.sub(r'(K:[^\n]+\n)', r'\1Q:1/4=120\n', abc_text)

    # Fix tricky repeats
    abc_text = re.sub(r'\|\:\s*\|\:', '|', abc_text)

    if not abc_text.endswith('|'):
        abc_text += ' |'

    return abc_text


def convert_abc_to_audio(abc_text, filename_base, attempts=3):
    """ABC → MIDI → WAV, with retry logic"""
    output_dir = "audio_output"
    os.makedirs(output_dir, exist_ok=True)

    processed_abc = fix_abc_for_audio(abc_text)

    abc_file = f"{output_dir}/{filename_base}.abc"
    with open(abc_file, "w") as f:
        f.write(processed_abc)

    wav_file = f"{output_dir}/{filename_base}.wav"

    for attempt in range(attempts):
        try:
            print(f"Converting {filename_base} to WAV...")
            os.system(f"abc2midi {abc_file} -o {output_dir}/{filename_base}.mid")
            os.system(f"timidity {output_dir}/{filename_base}.mid -Ow -o {wav_file}")

            if os.path.exists(wav_file):
                print(f"Created WAV file: {wav_file}")
                return wav_file
            else:
                print(f"Attempt {attempt+1}: Failed to create WAV file")
        except Exception as e:
            print(f"Attempt {attempt+1}: Error processing {filename_base}: {e}")

    print(f"Failed to convert {filename_base} after {attempts} attempts")
    return None


# === MAIN: generate N random songs with trained GRU ===
def main(num_songs_to_generate=8):
    print(" Loading dataset and GRU model...")
    import mitdeeplearning as mdl
    songs = mdl.lab1.load_training_data()

    # True vocab from songs (same as training)
    vocab = sorted(set("".join(songs)))
    print(f"Original vocab size: {len(vocab)}")

    char2idx = {u: i for i, u in enumerate(vocab)}
    idx2char = np.array(vocab)
    vocab_size = len(vocab)
    print(f"Using vocab size: {vocab_size}")

    # Create GRU model and load trained weights
    seq_length = 100
    gru_model = create_music_generation_model(vocab_size=vocab_size, use_gpu=True)

    try:
        dummy_input = tf.zeros((1, seq_length), dtype=tf.int32)
        _ = gru_model(dummy_input)  # build model
        gru_model.load_weights("/content/training_checkpoints/my_ckpt_final.weights.h5")
        print(" GRU model loaded successfully.")
    except Exception as e:
        print(f" Failed to load GRU model: {e}")
        return

    os.makedirs("audio_output", exist_ok=True)

    generated_songs_list = []
    for i in range(num_songs_to_generate):
        print(f"\n============== Generating Song {i+1}/{num_songs_to_generate} ==============")

        # PURE random generation (no explicit seed), only header + model’s learned distribution
        generated_song = generate_long_song_by_chaining(
            gru_model,
            char2idx,
            idx2char,
            target_length=500,
            seq_length=seq_length,
            temperature=1.0,
        )
        generated_songs_list.append(generated_song)

        # Save and convert to audio
        filename_base = f"generated_song_{i+1}"
        with open(f"audio_output/{filename_base}.abc", "w") as f:
            f.write(generated_song)

        audio_file = convert_abc_to_audio(generated_song, filename_base)

        if audio_file:
            print(f"\n Playing generated song {i+1}:")
            display(Audio(filename=audio_file))

    print(f"\nSuccessfully generated and played {num_songs_to_generate} random songs!")


# Run the main function to generate 8 songs
main(num_songs_to_generate=8)